In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [4]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

token = os.getenv("GEMINI_API_KEY")
endpoint = "https://generativelanguage.googleapis.com/v1beta/openai/"
model='gemini-2.5-flash'

openai_client = OpenAI(
    base_url=endpoint,
    api_key=token,
)

In [5]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [13]:
rec

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [6]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join the course. However, if you want to receive a certificate, you will need to submit your project while submissions are still being accepted.'

In [7]:
assistant.total_cost()

0.00213975

In [8]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [9]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course. However, if you want to receive a certificate, you will need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [14]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [15]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [16]:
assistant.reset_usage()